## Imports and Helper Functions

In [ ]:
import os
import glob
import cv2
import czifile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from cellpose import models
from skimage import exposure, morphology
from skimage.measure import label, regionprops

USE_GPU = False
model = models.Cellpose(gpu=USE_GPU, model_type="cyto")

def normalize_image(image):
    """Normalize an image to [0, 1]."""
    if image.size == 0:
        return image
    min_val, max_val = np.min(image), np.max(image)
    return image if max_val - min_val == 0 else (image - min_val) / (max_val - min_val)

def increase_contrast(image):
    """Stretch contrast for display or preprocessing."""
    p2, p98 = np.percentile(image, (2, 98))
    return exposure.rescale_intensity(image, in_range=(p2, p98))

In [ ]:
def segment_cells(green_channel):
    """Segment cells in the green channel."""
    green_channel = normalize_image(increase_contrast(green_channel))
    padding = 50
    padded_img = np.pad(green_channel, ((padding, padding), (padding, padding)), mode="constant", constant_values=0)

    diameter = 150
    while diameter < 700:
        masks, _, _, _ = model.eval(padded_img, diameter=diameter, channels=[0, 0])
        labeled_cells = label(masks)

        if np.max(labeled_cells) > 0:
            return labeled_cells[padding:-padding, padding:-padding]
        diameter += 50

    return None

def get_yellow_inclusion_mask(image_rgb):
    """Detect yellow inclusions in HSV space."""
    hsv_img = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    lower_yellow = np.array([15, 50, 50])
    upper_yellow = np.array([35, 255, 255])
    mask = cv2.inRange(hsv_img, lower_yellow, upper_yellow)
    return morphology.remove_small_objects(mask.astype(bool), min_size=5)

def get_bright_spots_mask(image_channel, spot_size_limit=25):
    """Detect distinct bright white spots."""
    p99 = np.percentile(image_channel, 99.9)
    img_clipped = np.clip(image_channel, 0, p99)
    denom = img_clipped.max() - img_clipped.min()
    img_norm = np.zeros_like(img_clipped) if denom == 0 else (img_clipped - img_clipped.min()) / denom

    selem = morphology.disk(spot_size_limit)
    spot_image = morphology.white_tophat(img_norm, selem)

    CONTRAST_THRESHOLD = 0.60
    MIN_BRIGHTNESS = 0.40

    mask = (spot_image > CONTRAST_THRESHOLD) & (img_norm > MIN_BRIGHTNESS)
    mask = morphology.remove_small_objects(mask, min_size=5)

    return mask, spot_image

def generate_inclusion_image_robust(green_channel, labeled_cells):
    """Build a robust inclusion map inside cells."""
    global_spot_mask, _ = get_bright_spots_mask(green_channel)
    final_inclusion_map = np.zeros_like(global_spot_mask, dtype=bool)

    for region in regionprops(labeled_cells):
        if region.area < 100:
            continue
        cell_mask = labeled_cells == region.label
        final_inclusion_map |= global_spot_mask & cell_mask

    return final_inclusion_map, global_spot_mask

In [ ]:
def get_yellow_inclusion_mask(image_rgb):
    """Detect yellow inclusions in HSV space."""
    hsv_img = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    lower_yellow = np.array([15, 50, 50])
    upper_yellow = np.array([35, 255, 255])
    mask = cv2.inRange(hsv_img, lower_yellow, upper_yellow)
    return morphology.remove_small_objects(mask.astype(bool), min_size=5)

def get_bright_spots_mask(image_channel, spot_size_limit=25):
    """Detect distinct bright white spots."""
    p99 = np.percentile(image_channel, 99.9)
    img_clipped = np.clip(image_channel, 0, p99)
    denom = img_clipped.max() - img_clipped.min()
    img_norm = np.zeros_like(img_clipped) if denom == 0 else (img_clipped - img_clipped.min()) / denom

    selem = morphology.disk(spot_size_limit)
    spot_image = morphology.white_tophat(img_norm, selem)

    CONTRAST_THRESHOLD = 0.60
    MIN_BRIGHTNESS = 0.40

    mask = (spot_image > CONTRAST_THRESHOLD) & (img_norm > MIN_BRIGHTNESS)
    mask = morphology.remove_small_objects(mask, min_size=5)

    return mask, spot_image

def generate_inclusion_image_robust(green_channel, labeled_cells):
    """Build a robust inclusion map inside cells."""
    global_spot_mask, _ = get_bright_spots_mask(green_channel)
    final_inclusion_map = np.zeros_like(global_spot_mask, dtype=bool)

    for region in regionprops(labeled_cells):
        if region.area < 100:
            continue
        cell_mask = labeled_cells == region.label
        final_inclusion_map |= global_spot_mask & cell_mask

    return final_inclusion_map, global_spot_mask

In [ ]:
def get_bright_spots_mask(image_channel, spot_size_limit=25):
    """Detect distinct bright white spots."""
    p99 = np.percentile(image_channel, 99.9)
    img_clipped = np.clip(image_channel, 0, p99)
    denom = img_clipped.max() - img_clipped.min()
    img_norm = np.zeros_like(img_clipped) if denom == 0 else (img_clipped - img_clipped.min()) / denom

    selem = morphology.disk(spot_size_limit)
    spot_image = morphology.white_tophat(img_norm, selem)

    CONTRAST_THRESHOLD = 0.60
    MIN_BRIGHTNESS = 0.40

    mask = (spot_image > CONTRAST_THRESHOLD) & (img_norm > MIN_BRIGHTNESS)
    mask = morphology.remove_small_objects(mask, min_size=5)

    return mask, spot_image

def generate_inclusion_image_robust(green_channel, labeled_cells):
    """Build a robust inclusion map inside cells."""
    global_spot_mask, _ = get_bright_spots_mask(green_channel)
    final_inclusion_map = np.zeros_like(global_spot_mask, dtype=bool)

    for region in regionprops(labeled_cells):
        if region.area < 100:
            continue
        cell_mask = labeled_cells == region.label
        final_inclusion_map |= global_spot_mask & cell_mask

    return final_inclusion_map, global_spot_mask

In [ ]:
# Install czifile if needed: pip install czifile

## TMEM175 Inclusion Analysis - Day 1

In [ ]:
IMAGE_FOLDER = os.path.join(os.getcwd(), "090425_TMEM175_INCLUSIONS_day 1")
FILE_EXTENSION = "*.czi"
RESULTS_FOLDER = os.path.join(os.getcwd(), "Results")
MASK_FOLDER = os.path.join(RESULTS_FOLDER, "masks")

os.makedirs(RESULTS_FOLDER, exist_ok=True)
os.makedirs(MASK_FOLDER, exist_ok=True)

print(f"Looking in: {IMAGE_FOLDER}")

image_files = glob.glob(os.path.join(IMAGE_FOLDER, FILE_EXTENSION))
results = []

for filepath in image_files:
    filename = os.path.basename(filepath)
    print(f"Processing: {filename}...")

    try:
        img_data = czifile.imread(filepath).squeeze()
    except Exception as e:
        print(f"  Error: {e}")
        continue

    if img_data.ndim == 3:
        green_channel = img_data[1]
    else:
        green_channel = img_data

    green_norm = normalize_image(green_channel)
    green_uint8 = (green_norm * 255).astype(np.uint8)

    labeled_cells = segment_cells(green_uint8)
    if labeled_cells is None:
        print("  No cells found.")
        continue

    inclusion_mask, global_debug_mask = generate_inclusion_image_robust(green_norm, labeled_cells)
    num_inclusions = label(inclusion_mask).max()
    print(f"  -> Found {num_inclusions} inclusions.")

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(green_norm, cmap="gray")
    axes[0].set_title("Original Green Channel")

    axes[1].imshow(global_debug_mask, cmap="gray")
    axes[1].set_title("Raw Bright Spots (Top-Hat)")

    axes[2].imshow(labeled_cells, cmap="nipy_spectral")
    axes[2].set_title("Cell Masks")

    axes[3].imshow(green_norm, cmap="gray")
    axes[3].imshow(inclusion_mask, cmap="autumn", alpha=0.6)
    axes[3].set_title(f"Final Count: {num_inclusions}")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    results.append({"filename": filename, "inclusions": num_inclusions})

if results:
    print(pd.DataFrame(results).head())